In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.ensemble import StackingClassifier
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

def downsample(X, y, T):
    def custom_sampling_strategy(cls_counts):
        if not isinstance(cls_counts, dict):
            cls_counts = dict(Counter(cls_counts))
        return {cls: min(count, T) for cls, count in cls_counts.items()}

    # down sample labels that are significantly more than others
    undersampler = RandomUnderSampler(sampling_strategy=custom_sampling_strategy, random_state=42)
    X, y = undersampler.fit_resample(X, y)
    return X, y

def upsample(X, y, T, use_smote=False):
    def smote_strategy(cls_counts):
        cls_counts = dict(Counter(cls_counts))
        return {cls: T for cls, count in cls_counts.items() if count < T}

    if not use_smote:
        # over sample minority labels by duplication
        oversampler = RandomOverSampler(random_state=42)
        X, y = oversampler.fit_resample(X, y)
    else:
        # over sample minority labels by SMOTE
        smote = SMOTE(sampling_strategy=smote_strategy, k_neighbors=2, random_state=42)
        X, y = smote.fit_resample(X, y)
    return X, y

def dt_train(X, y, cw=None):
    # setup KFold validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # setup decision tree with cross validation
    dt_param_grid = {
        'criterion': ['entropy', 'gini'],
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10]
    }
    dt_model = DecisionTreeClassifier(random_state=42, class_weight=cw)
    dt_grid = GridSearchCV(dt_model, dt_param_grid, cv=kf, scoring='accuracy', n_jobs=-1)
    dt_grid.fit(X, y)
    dt_best = dt_grid.best_estimator_
    return dt_grid, dt_best

def lr_train(X, y, cw=None):
    # setup KFold validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # setup logistic regression with cross validation
    C_values = np.logspace(-2,2,10)
    lr_param_grid = {
        'C': C_values,
    }

    lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight=cw)
    lr_grid = GridSearchCV(lr_model, lr_param_grid, cv=kf, scoring='accuracy', n_jobs=4)
    lr_grid.fit(X_train, y_train)
    lr_best = lr_grid.best_estimator_
    return lr_grid, lr_best

def svm_train(X, y, cw=None):
    # setup KFold validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # setup svm with cross validation
    C_values = np.logspace(-2,2,5)
    param_grid = [
        {'kernel': ['linear'], 'C': C_values},
        {'kernel': ['rbf'], 'C': C_values,
        'gamma': ['scale', 'auto']}
    ]

    svm_model = SVC(random_state=42, class_weight=cw)
    svm_grid = GridSearchCV(svm_model, param_grid, cv=kf, scoring='accuracy', n_jobs=4)
    svm_grid.fit(X_train, y_train)
    svm_best = svm_grid.best_estimator_
    return svm_grid, svm_best

def xgb_train(X, y):
    # setup KFold validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # setup xgb with cross validation
    param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [3, 5],
        'learning_rate': [0.1, 0.2],
        'subsample': [1.0],
        'colsample_bytree': [1.0],
        'gamma': [0, 0.1]
    }
    
    xgb_model = xgb.XGBClassifier(random_state=42)
    xgb_grid = GridSearchCV(xgb_model, param_grid, cv=kf, scoring='accuracy', n_jobs=4)
    xgb_grid.fit(X_train, y_train)
    xgb_best = xgb_grid.best_estimator_
    return xgb_grid, xgb_best

def stacking_train(models, f_estimator, X, y):
    # setup KFold validation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # setup stacking model with cross validation
    estimators = models
    stacking_model = StackingClassifier(estimators=estimators, final_estimator=f_estimator, cv=kf)
    stacking_model.fit(X, y)
    return stacking_model

def model_eval(model, X, y):
    # get classification_report
    preds = model.predict(X)
    print(classification_report(y, preds, zero_division=0))

    # get accuracy
    accuracy = accuracy_score(y, preds)
    # get precision
    precision = precision_score(y, preds, average='weighted', zero_division=0)
    # get recall
    recall = recall_score(y, preds, average='weighted', zero_division=0)
    # get f1
    f1 = f1_score(y, preds, average='weighted', zero_division=0)
    return accuracy, precision, recall, f1

def plot_cm(y, preds):
    # plot confusion_matrix
    cm = confusion_matrix(y, preds)
    plt.figure(figsize=(12, 12)) 
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=set(y), yticklabels=set(y))
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.show()

def train_eval(X_train, y_train, X_test, y_test, filename, cw=None):
    eval_entries = []
    # train and evaluate models
    dt_model, dt_best = dt_train(X_train, y_train, cw=cw)
    accuracy, precision, recall, f1 = model_eval(dt_model, X_test, y_test)
    eval_entries.append(['Decision Tree', accuracy, precision, recall, f1])

    lr_model, lr_best = lr_train(X_train, y_train, cw=cw)
    accuracy, precision, recall, f1 = model_eval(lr_model, X_test, y_test)
    eval_entries.append(['Logistic Regression', accuracy, precision, recall, f1])

    svm_model, svm_best = svm_train(X_train, y_train, cw=cw)
    accuracy, precision, recall, f1 = model_eval(svm_model, X_test, y_test)
    eval_entries.append(['SVM', accuracy, precision, recall, f1])

    xgb_model, xgb_best = xgb_train(X_train, y_train)
    accuracy, precision, recall, f1 = model_eval(xgb_model, X_test, y_test)
    eval_entries.append(['XGBoost', accuracy, precision, recall, f1])

    # stacking technique
    models = [
        ('dt', dt_best),
        ('svm', svm_best),
        ('xgb', xgb_best)
    ]
    stacking_model = stacking_train(models, lr_best, X_train, y_train)
    accuracy, precision, recall, f1 = model_eval(stacking_model, X_test, y_test)
    eval_entries.append(['Stacking', accuracy, precision, recall, f1])

    # save evaluation results
    eval_df = pd.DataFrame(eval_entries, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1'])
    eval_df.to_csv(filename, index=False)

if __name__ == '__main__':
    # read data
    X_data = pd.read_csv('X_train_cleaned.csv')
    y_data = pd.read_csv('y_train_cleaned.csv')
    y_data = y_data.values.ravel()

    # scale features
    scaler = StandardScaler()
    X_data_scaled = scaler.fit_transform(X_data)

    # split train, test data
    X_train, X_test, y_train, y_test = train_test_split(X_data_scaled, y_data, test_size=0.2, random_state=42, stratify=y_data)
    
 
    # resample training data to increase data balance
    count_dict = Counter(y_train)
    print('Original data distribution: ', count_dict)

    

    # upsample only
    X_train, y_train = upsample(X_train, y_train, T=300, use_smote=True)

    # data after downsampling
    print('New data distribution: ', dict(Counter(y_train)))
    print(X_train.shape[0])

   
    train_eval(X_train, y_train, X_test, y_test, filename='eval_upsample_costsensitive.csv', cw='balanced')


Original data distribution:  Counter({5: 3497, 10: 834, 6: 424, 8: 392, 12: 354, 24: 298, 17: 274, 26: 218, 21: 209, 14: 205, 4: 182, 25: 141, 19: 137, 20: 119, 27: 84, 7: 82, 11: 50, 3: 50, 18: 47, 13: 46, 23: 33, 15: 21, 9: 20, 0: 14, 1: 6, 22: 5, 16: 5, 2: 5})
New data distribution:  {6: 424, 5: 3497, 7: 300, 10: 834, 14: 300, 4: 300, 24: 300, 8: 392, 13: 300, 12: 354, 1: 300, 26: 300, 17: 300, 21: 300, 20: 300, 25: 300, 0: 300, 27: 300, 11: 300, 3: 300, 19: 300, 23: 300, 18: 300, 9: 300, 22: 300, 15: 300, 16: 300, 2: 300}
12401
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.12      0.23      0.16        13
           4       0.21      0.26      0.24        46
           5       0.81      0.77      0.79       875
           6       0.64      0.58      0.61       106
           7       0.11      

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.43      0.46      0.44        13
           4       0.45      0.63      0.53        46
           5       0.96      0.76      0.85       875
           6       0.87      0.90      0.88       106
           7       0.44      0.57      0.50        21
           8       0.77      0.64      0.70        98
           9       0.25      0.20      0.22         5
          10       0.77      0.66      0.71       208
          11       0.56      0.75      0.64        12
          12       0.56      0.50      0.53        88
          13       0.00      0.00      0.00        12
          14       0.13      0.50      0.21        52
          15       0.67      0.80      0.73         5
          16       0.00      0.00      0.00         1
          17       0.68    

/Users/davinagreen/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         2
           3       0.38      0.23      0.29        13
           4       0.63      0.72      0.67        46
           5       0.94      0.88      0.91       875
           6       0.87      0.94      0.90       106
           7       0.50      0.38      0.43        21
           8       0.78      0.80      0.79        98
           9       0.00      0.00      0.00         5
          10       0.76      0.80      0.78       208
          11       0.53      0.67      0.59        12
          12       0.51      0.57      0.53        88
          13       0.00      0.00      0.00        12
          14       0.17      0.21      0.19        52
          15       0.71      1.00      0.83         5
          16       0.00      0.00      0.00         1
          17       0.64    